In [36]:
import pandas as pd
from statsmodels.stats.multitest import multipletests
from config import RF_PERFORMANCE,GRANGER_RESULTS, LASSO_RESULTS,RF_RESULTS,RUNTIME_LOG

In [8]:

perf = pd.read_csv(RF_PERFORMANCE)

In [9]:
tbl = (perf[["model", "n_features", "r2_oos_ct", "mae_oos", "dir_acc_oos"]]
       .round({"r2_oos_ct": 4, "mae_oos": 5, "dir_acc_oos": 4})
       .rename(columns={"model": "Specification",
                        "n_features": "Features",
                        "r2_oos_ct": "OOS R²",
                        "mae_oos": "MAE",
                        "dir_acc_oos": "Dir. accuracy"}))
tbl.to_clipboard(index=False)

In [10]:
tbl

,Specification,Features,OOS R²,MAE,Dir. accuracy
0,A_baseline,7,-0.0034,0.01556,0.5016
1,B_all,15,-0.0264,0.01591,0.4838
2,C_granger,13,-0.0325,0.01595,0.4994
3,D_lasso_path,9,-0.0501,0.01623,0.5009


In [19]:
gr = pd.read_parquet(GRANGER_RESULTS)

In [23]:
gr = pd.read_parquet(GRANGER_RESULTS).reset_index()

rates = (gr.assign(rej_1pct=gr["p_value"] < 0.01,
                   rej_5pct=gr["p_value"] < 0.05,
                   rej_10pct=gr["p_value"] < 0.10)
           .groupby("test")
           .agg(n_tests=("p_value", "size"),
                rej_1pct=("rej_1pct", "mean"),
                rej_5pct=("rej_5pct", "mean"),
                rej_10pct=("rej_10pct", "mean")))

for c in ["rej_1pct", "rej_5pct", "rej_10pct"]:
    rates[c] = (rates[c] ).round(3)

rates = rates.sort_values("rej_5pct", ascending=False)
rates.to_clipboard()
print(rates)

                           n_tests  rej_1pct  rej_5pct  rej_10pct
test                                                             
universe_sentiment_volume     1390     0.059     0.235      0.385
joint_all_sentiment           1390     0.052     0.160      0.258
sentiment_volume_z            1390     0.028     0.097      0.171
sentiment_score_z             1390     0.022     0.069      0.119
universe_sentiment_score      1390     0.001     0.013      0.042


In [25]:
lasso = pd.read_parquet(LASSO_RESULTS)

In [26]:
print(lasso.columns.tolist()); print(lasso.head(10)); print(len(lasso))

['run', 'feature', 'coefficient', 'selected', 'entry_alpha', 'path_rank', 'alpha', 'n_rows', 'n_features', 'runtime_sec', 'prep_runtime_sec']
                          run                         feature   coefficient  \
0  Run A (Granger comparison)                              r0 -0.000000e+00   
1  Run A (Granger comparison)                             r_1  3.255635e-17   
2  Run A (Granger comparison)               sentiment_score_z  0.000000e+00   
3  Run A (Granger comparison)              sentiment_volume_z -0.000000e+00   
4  Run A (Granger comparison)        universe_sentiment_score  0.000000e+00   
5  Run A (Granger comparison)       universe_sentiment_volume  0.000000e+00   
6  Run A (Granger comparison)          sentiment_score_z_lag2 -0.000000e+00   
7  Run A (Granger comparison)         sentiment_volume_z_lag2 -0.000000e+00   
8  Run A (Granger comparison)   universe_sentiment_score_lag2  0.000000e+00   
9  Run A (Granger comparison)  universe_sentiment_volume_lag2 -0.000

In [28]:
tbl = lasso.pivot(index="feature", columns="run", values="path_rank")
tbl.columns = [c.split(" (")[0] for c in tbl.columns]        # -> "Run A", "Run B"
tbl = (tbl.astype("Int64")                                    # whole-number ranks, keeps blanks
          .rename(columns={"Run A": "Rank (Run A)", "Run B": "Rank (Run B)"})
          .sort_values("Rank (Run B)")
          .reset_index()
          .rename(columns={"feature": "Feature"}))
tbl.to_clipboard(index=False)
print(tbl)

                           Feature  Rank (Run A)  Rank (Run B)
0                              r_1             1             1
1   universe_sentiment_volume_lag2             2             2
2                               r0             3             3
3        universe_sentiment_volume             4             4
4    universe_sentiment_score_lag2             5             5
5                            r_5_2          <NA>             6
6                           r0_z20          <NA>             7
7         universe_sentiment_score             6             8
8           sentiment_score_z_lag2             8             9
9                sentiment_score_z             7            10
10                          r_10_6          <NA>            11
11              sentiment_volume_z             9            12
12         sentiment_volume_z_lag2            10            13
13                      Sector_z20          <NA>            14
14                            pvma          <NA>       

In [30]:
rf = pd.read_parquet(RF_RESULTS)

In [33]:
# Rank features within each model by permutation importance (1 = most important)
rf["rank"] = (rf.groupby("model")["importance_permutation"]
                .rank(ascending=False, method="first")
                .astype(int))

grid = (rf.pivot(index="feature", columns="model", values="rank")
          .astype("Int64"))                     # keeps blanks as <NA>, ranks as whole numbers

grid = grid[["A_baseline", "B_all", "C_granger", "D_lasso_path"]]  # column order
grid = grid.sort_values("B_all").reset_index()

# Report-friendly labels
grid["feature"] = (grid["feature"]
                   .str.replace(r"_lag1$", " (lag 1)", regex=True)
                   .str.replace(r"_lag2$", " (lag 2)", regex=True))

sent_vars = ["sentiment_score_z", "sentiment_volume_z",
             "universe_sentiment_score", "universe_sentiment_volume"]
grid["feature"] = grid["feature"].apply(
    lambda f: f + " (lag 1)" if f in sent_vars else f)

grid.columns = ["Feature", "A – baseline", "B – all sentiment",
                "C – Granger-selected", "D – LASSO-selected"]
grid.to_clipboard(index=False)
print(grid.to_string(index=False))

                          Feature  A – baseline  B – all sentiment  C – Granger-selected  D – LASSO-selected
                       Sector_z20             1                  1                     1                   1
                           r0_z20             2                  2                     2                   3
                               r0             7                  3                     6                   7
                            r_5_2             4                  4                     4                   4
                           r_10_6             6                  5                    11                   6
 universe_sentiment_score (lag 1)          <NA>                  6                  <NA>                <NA>
universe_sentiment_volume (lag 2)          <NA>                  7                    13                   9
        sentiment_score_z (lag 1)          <NA>                  8                     8                <NA>
       sentiment_vo

In [35]:
rf.query("model=='B_all'")[["feature","importance_permutation","perm_std"]] \
  .sort_values("importance_permutation", ascending=False).to_clipboard(index=False)

In [37]:
run = pd.read_csv(RUNTIME_LOG)

In [39]:
run.head().to_clipboard(index=False)

In [40]:
lasso = pd.read_parquet(LASSO_RESULTS)
print(lasso.groupby("run")[["prep_runtime_sec", "runtime_sec"]].first())

                             prep_runtime_sec  runtime_sec
run                                                       
Run A (Granger comparison)           5.124792     2.581985
Run B (All pricing features          5.124792     3.192105


In [43]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r"P:/Personal/Birkbeck/MSc Project/git_msc_project_a/data/processed")

FILES = {
    "fwd_r1":     "rf_performance_1.csv",
    "fwd_r3":     "rf_performance_1_fwd_r3.csv",
    "fwd_r5":     "rf_performance_1_fwd_r5.csv",
    "fwd_r1_sec": "rf_performance_1_fwd_r1_sec.csv",
    "fwd_r3_sec": "rf_performance_1_fwd_r3_sec.csv",
    "fwd_r5_sec": "rf_performance_1_fwd_r5_sec.csv",
}

frames = []
for target, fname in FILES.items():
    d = pd.read_csv(DATA_DIR / fname)
    d["target"] = target          # base file has no tag; harmless overwrite for the rest
    frames.append(d)

perf = pd.concat(frames, ignore_index=True)

# --- the 5.5 grid: models as rows, targets as columns, OOS R² in cells -----
r2_grid = (perf.pivot(index="model", columns="target", values="r2_oos_ct")
               .loc[["A_baseline", "B_all", "C_granger", "D_lasso_path"],
                    ["fwd_r1", "fwd_r3", "fwd_r5",
                     "fwd_r1_sec", "fwd_r3_sec", "fwd_r5_sec"]]
               .round(4))

r2_grid.to_clipboard()
print(r2_grid.to_string())

target        fwd_r1  fwd_r3  fwd_r5  fwd_r1_sec  fwd_r3_sec  fwd_r5_sec
model                                                                   
A_baseline   -0.0034 -0.0087 -0.0104     -0.0016     -0.0027     -0.0023
B_all        -0.0264 -0.0299 -0.0264     -0.0010     -0.0024     -0.0022
C_granger    -0.0325 -0.0364 -0.0201     -0.0018     -0.0026     -0.0030
D_lasso_path -0.0501 -0.0531 -0.0320     -0.0039     -0.0038     -0.0043


In [42]:
# Directional accuracy grid, same shape (worth eyeballing even if only R² goes in 5.5)
dir_grid = perf.pivot(index="model", columns="target", values="dir_acc_oos").round(4)
print((dir_grid * 100).round(1).to_string())

# Benchmarks per target, for the table's benchmark row / footnote
print(perf.groupby("target")[["bench_mae_oos", "bench_dir_acc"]].first().round(5))

target        fwd_r1  fwd_r1_sec  fwd_r3  fwd_r3_sec  fwd_r5  fwd_r5_sec
model                                                                   
A_baseline      50.2        50.1    50.1        50.1    49.9        50.2
B_all           48.4        50.0    49.6        49.9    49.8        50.2
C_granger       49.9        50.1    50.2        50.1    50.7        50.0
D_lasso_path    50.1        50.1    50.4        50.1    50.9        50.0
            bench_mae_oos  bench_dir_acc
target                                  
fwd_r1            0.01551        0.50719
fwd_r1_sec        0.01253        0.49589
fwd_r3            0.02790        0.51610
fwd_r3_sec        0.02254        0.49604
fwd_r5            0.03654        0.51945
fwd_r5_sec        0.02965        0.50432
